# Data Validation, Cleaning & Score Column Removal

**Plan 2 - Tasks 1 & 2**

Validate data integrity, handle quality issues, drop score columns (per proposal feedback 2026-04-30).

## Step 1: Load Dataset and Verify Shape

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../datasets/student_exam_performance_dataset.csv')
print(f"Shape: {df.shape}")
assert df.shape == (10000, 23), "Unexpected shape"
print("Shape validation: PASSED")

Shape: (10000, 23)
Shape validation: PASSED


## Step 2: Check for Missing Values

In [2]:
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.sum() > 0 else "No missing values found")

Missing values per column:
No missing values found


## Step 3: Check for Duplicate Records

In [3]:
duplicates = df.duplicated().sum()
print(f"Duplicate rows: {duplicates}")
dup_ids = df['student_id'].duplicated().sum()
print(f"Duplicate student IDs: {dup_ids}")

Duplicate rows: 0
Duplicate student IDs: 0


## Step 4: Validate Categorical Column Values

In [4]:
expected = {
    'gender': {'Male', 'Female'},
    'parental_education': {'High School', 'Bachelor', 'Master', 'PhD'},
    'family_income': {'Low', 'Medium', 'High'},
    'internet_access': {'Yes', 'No'},
    'study_environment': {'Quiet', 'Moderate', 'Noisy'},
    'tutoring': {'Yes', 'No'},
    'pass_fail': {'Pass', 'Fail'},
    'grade_category': {'A', 'B', 'C', 'D', 'F'}
}
for col, vals in expected.items():
    actual = set(df[col].unique())
    unexpected = actual - vals
    if unexpected:
        print(f"WARNING: {col} has unexpected values: {unexpected}")
    else:
        print(f"{col}: OK ({len(actual)} valid categories)")

gender: OK (2 valid categories)
parental_education: OK (4 valid categories)
family_income: OK (3 valid categories)
internet_access: OK (2 valid categories)
study_environment: OK (3 valid categories)
tutoring: OK (2 valid categories)
pass_fail: OK (2 valid categories)
grade_category: OK (5 valid categories)


## Step 5: Validate Numeric Column Ranges

In [5]:
range_checks = {
    'age': (15, 18),
    'study_hours_per_day': (0, 24),
    'attendance_rate': (0, 100),
    'sleep_hours': (0, 24),
    'social_media_hours': (0, 24),
    'assignment_completion_rate': (0, 100),
    'participation_score': (0, 100),
    'math_score': (0, 100),
    'reading_score': (0, 100),
    'writing_score': (0, 100),
    'science_score': (0, 100),
    'final_exam_score': (0, 100),
    'previous_gpa': (0, 4.0)
}
for col, (lo, hi) in range_checks.items():
    oob = ((df[col] < lo) | (df[col] > hi)).sum()
    if oob > 0:
        print(f"WARNING: {col} has {oob} out-of-range values [{lo}, {hi}]")
    else:
        print(f"{col}: OK (range [{df[col].min():.2f}, {df[col].max():.2f}])")

age: OK (range [15.00, 18.00])
study_hours_per_day: OK (range [0.50, 7.24])
attendance_rate: OK (range [50.80, 100.00])
sleep_hours: OK (range [4.00, 10.00])
social_media_hours: OK (range [0.00, 8.00])
assignment_completion_rate: OK (range [40.00, 100.00])
participation_score: OK (range [20.00, 100.00])
math_score: OK (range [0.00, 100.00])
reading_score: OK (range [0.00, 100.00])
writing_score: OK (range [0.00, 100.00])
science_score: OK (range [4.80, 100.00])
final_exam_score: OK (range [4.40, 97.80])
previous_gpa: OK (range [0.00, 3.99])


## Step 5b: IQR Outlier Detection

Detect outliers via the **IQR method** (1.5 × IQR rule) on numeric columns. Following the cleaning decision, outliers are **retained** as valid behavioral variation rather than data-entry errors.

In [6]:
# Step 5b: IQR-based Outlier Detection (1.5 x IQR rule). Outliers are RETAINED as valid data.
numeric_cols = ['age', 'study_hours_per_day', 'attendance_rate', 'sleep_hours',
                'social_media_hours', 'assignment_completion_rate',
                'online_courses_completed', 'final_exam_score', 'previous_gpa']
print("IQR outlier detection — outliers RETAINED (valid behavioral variation, not errors):")
total_outliers = 0
for col in numeric_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_out = int(((df[col] < lower) | (df[col] > upper)).sum())
    total_outliers += n_out
    print(f"  {col}: {n_out if n_out else 'none'} outliers (bounds [{lower:.2f}, {upper:.2f}])")
print(f"\nTotal outlier flags: {total_outliers}. Decision: RETAIN (valid data, not removed).")

IQR outlier detection — outliers RETAINED (valid behavioral variation, not errors):
  age: none outliers (bounds [12.00, 20.00])
  study_hours_per_day: 27 outliers (bounds [-0.24, 6.28])
  attendance_rate: 36 outliers (bounds [57.84, 112.34])
  sleep_hours: 63 outliers (bounds [4.31, 9.72])
  social_media_hours: 36 outliers (bounds [-1.50, 6.50])
  assignment_completion_rate: 30 outliers (bounds [40.50, 119.70])
  online_courses_completed: 49 outliers (bounds [-2.00, 6.00])
  final_exam_score: 95 outliers (bounds [17.60, 81.60])
  previous_gpa: 62 outliers (bounds [0.50, 3.46])

Total outlier flags: 398. Decision: RETAIN (valid data, not removed).


## Step 6: Standardize String Columns

In [7]:
string_cols = df.select_dtypes(include='object').columns
for col in string_cols:
    if col != 'student_id':
        df[col] = df[col].str.strip().str.title()
print(f"Standardized {len(string_cols) - 1} string columns")

Standardized 8 string columns


## Step 7: Verify Grade/Pass-Fail Consistency

In [8]:
inconsistent_mask = ((df['grade_category'] == 'F') & (df['pass_fail'] == 'Pass')) | \
                    ((df['grade_category'] != 'F') & (df['pass_fail'] == 'Fail'))
n_inconsistent = int(inconsistent_mask.sum())
print(f"Inconsistent pass_fail/grade_category rows: {n_inconsistent}")

# Investigate the disagreement: report the final_exam_score profile of these rows.
if n_inconsistent > 0:
    s = df.loc[inconsistent_mask, 'final_exam_score']
    print(f"  final_exam_score of inconsistent rows -> min={s.min():.1f}, "
          f"max={s.max():.1f}, mean={s.mean():.1f}")
    print("  Label disagreement between letter-grade and pass/fail (boundary cases).")

# Decision (DN2): RETAIN all rows. pass_fail is the modeling target and is internally
# valid on its own; grade_category is used only descriptively. Dropping would shrink
# N below 10,000 and cascade through every downstream notebook. Kept for traceability.
print(f"Decision: RETAIN all {len(df)} rows (label disagreement, not data corruption).")

Inconsistent pass_fail/grade_category rows: 30
  final_exam_score of inconsistent rows -> min=50.0, max=50.0, mean=50.0
  Label disagreement between letter-grade and pass/fail (boundary cases).
Decision: RETAIN all 10000 rows (label disagreement, not data corruption).


## Step 8: Drop Score Columns (Proposal Feedback 2026-04-30)

Research focuses on **student behavior** predicting  and . Internal score columns are removed to isolate behavioral factors.

In [9]:
score_cols_to_drop = ['math_score', 'reading_score', 'writing_score',
                       'science_score', 'participation_score']
print(f"Columns before drop: {df.shape[1]}")
print(f"Dropping score columns: {score_cols_to_drop}")
df = df.drop(columns=score_cols_to_drop)
print(f"Columns after drop: {df.shape[1]}")
print(f"Remaining columns: {list(df.columns)}")

Columns before drop: 23
Dropping score columns: ['math_score', 'reading_score', 'writing_score', 'science_score', 'participation_score']
Columns after drop: 18
Remaining columns: ['student_id', 'gender', 'age', 'parental_education', 'family_income', 'internet_access', 'study_environment', 'study_hours_per_day', 'attendance_rate', 'sleep_hours', 'social_media_hours', 'assignment_completion_rate', 'online_courses_completed', 'tutoring', 'final_exam_score', 'previous_gpa', 'pass_fail', 'grade_category']


## Step 9: Save Cleaned Dataset

In [10]:
df.to_csv('../datasets/student_exam_cleaned.csv', index=False)
print(f"Cleaned dataset saved: {df.shape}")
df.head()

Cleaned dataset saved: (10000, 18)


,student_id,gender,age,parental_education,family_income,internet_access,study_environment,study_hours_per_day,attendance_rate,sleep_hours,social_media_hours,assignment_completion_rate,online_courses_completed,tutoring,final_exam_score,previous_gpa,pass_fail,grade_category
0,S00001,Male,17,High School,Medium,Yes,Quiet,2.98,96.5,6.05,0.1,80.5,1,Yes,49.1,2.44,Fail,F
1,S00002,Female,18,High School,Low,Yes,Quiet,4.45,95.7,6.96,2.9,70.9,0,Yes,70.1,2.79,Pass,C
2,S00003,Male,17,High School,Medium,No,Quiet,3.75,76.0,7.02,2.4,77.6,4,Yes,42.2,1.49,Fail,F
3,S00004,Male,18,Bachelor,Medium,Yes,Quiet,2.03,72.6,6.23,3.5,63.5,4,No,31.9,1.34,Fail,F
4,S00005,Male,18,Bachelor,Medium,Yes,Quiet,5.14,87.3,8.54,2.1,71.8,0,No,66.4,2.60,Pass,C
